# 03 · Fine-tune DistilBERT on Google Colab (free T4 GPU)
**Runtime → Change runtime type → T4 GPU** before running.

Steps: get the code → get the data → install → train (~15-25 min) → evaluate → download `models/distilbert`.

In [ ]:
!nvidia-smi -L || echo "No GPU - switch runtime to T4 GPU (training on CPU takes hours)" 

## 1. Get the project code
Option A — clone your GitHub repo. Option B — upload the project zip.

In [ ]:
import os
REPO_URL = ""  # e.g. "https://github.com/<you>/fake-job-detector.git"
if REPO_URL:
    !git clone -q $REPO_URL fake-job-detector
else:
    from google.colab import files
    print("Upload fake-job-detector.zip")
    up = files.upload()
    zip_name = next(iter(up))
    !unzip -q -o "$zip_name" -d .
%cd fake-job-detector
!ls

## 2. Get the EMSCAD data
Option A — Kaggle API (upload your `kaggle.json`). Option B — upload `fake_job_postings.csv` directly.

In [ ]:
import os
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/fake_job_postings.csv"):
    from google.colab import files
    print("Upload kaggle.json (Kaggle > Settings > Create New Token) OR fake_job_postings.csv")
    up = files.upload()
    name = next(iter(up))
    if name.endswith(".json"):
        !mkdir -p ~/.kaggle && mv "$name" ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json
        !pip -q install kaggle
        !kaggle datasets download -d shivamb/real-or-fake-fake-jobposting-prediction -p data/raw --unzip
    else:
        !mv "$name" data/raw/fake_job_postings.csv
!ls -lh data/raw

## 3. Install dependencies

In [ ]:
# Colab already has torch; install the rest
!pip -q install -r requirements.txt
!pip -q install "transformers>=4.40" "accelerate>=0.30" 

## 4. Train
`--sample-frac 0.2` gives a ~5-minute sanity run. Use the full data for final results.

In [ ]:
!python -m src.train_bert --epochs 3 --max-len 256 --batch-size 16 --lr 3e-5

## 5. Compare with the classical models

In [ ]:
# Train the classical models too (fast) so the comparison table has all rows
!python -m src.train_classical --no-smote
!python -m src.evaluate --update-readme
import pandas as pd
df = pd.read_csv("results/model_comparison.csv")
df[["model", "dataset", "mode", "precision", "recall", "f1", "pr_auc", "accuracy"]].round(3)

## 6. Try it

In [ ]:
import sys; sys.path.insert(0, ".")
from src.predict import Predictor
p = Predictor(use_bert=True)
print(p.info())
out = p.predict("Earn ₹40,000 per week from home! Registration fee ₹1,499. WhatsApp HR 9000000001. No interview.")
{k: out[k] for k in ["label", "risk_score", "model_probability", "rule_score", "explanation_method"]}

## 7. Download the trained model
Unzip into your local `models/` folder, then run the app with `USE_BERT=1` (see README).

In [ ]:
!rm -rf models/distilbert_checkpoints
!zip -qr distilbert_model.zip models/distilbert models/*.joblib models/model_meta.json results
from google.colab import files
files.download("distilbert_model.zip")